# 05 Generate Healthy_001

## Objective
Run one bounded real Healthy baseline rollout and create the canonical dataset package.

## Prerequisites
Run notebook 00 in Python 3.12 with FlyGym 2.1.0 and MuJoCo 3.9.0.

## Expected Output
datasets/healthy/Healthy_001/ with manifest.json, rollout.json, rollout.npz, metadata.json, logs/, and a pipeline-compatible rollouts/rollout_arrays.npz.

## Troubleshooting
This notebook uses the existing official CPG controller helper. Stop if the real FlyGym assets or controller helper cannot be imported.

## Validation
The dataset manifest records files, checksums, schema, and the bounded rollout frame count.

## Next notebook
06_Inspect_Dataset.ipynb

In [ ]:
import hashlib
import json
import os
from pathlib import Path
import subprocess
import numpy as np
repo = Path.cwd() / 'drosophila-pd-flygym'
if (repo / 'pyproject.toml').is_file():
    os.chdir(repo)
try:
    from drosophila_pd.controllers.healthy_baseline import build_official_cpg_controller
    from drosophila_pd.experiments.healthy_baseline import load_healthy_baseline_config
    from drosophila_pd.flygym_adapter import FlyGymAdapter, FlyGymConfig, RolloutRecorder, export_rollout
    from flygym_demo.complex_terrain import LocomotionAction, apply_locomotion_action
    config = FlyGymConfig.from_yaml('configs/v2/flygym/healthy.yaml')
    baseline_config = load_healthy_baseline_config('configs/experiments/healthy_baseline.yaml')
    adapter = FlyGymAdapter()
    fly = adapter.create_fly(config.fly)
    world = adapter.create_world(config.world)
    adapter.attach_fly(world, fly, position=config.world.spawn_position, orientation=config.world.spawn_orientation, add_ground_contact_sensors=config.world.add_ground_contact_sensors)
    simulation = adapter.create_simulation(world, config.simulation)
    simulation.reset()
    dof_order = fly.get_actuated_jointdofs_order('position')
    controller, preprogrammed_steps = build_official_cpg_controller(timestep=simulation.timestep, random_seed=baseline_config.random_seed, output_dof_order=dof_order, config=baseline_config.controller)
    initial_action = LocomotionAction(joint_angles=preprogrammed_steps.default_pose_by_dof_order(dof_order), adhesion_onoff=np.ones(6, dtype=bool))
    apply_locomotion_action(simulation, fly.name, initial_action)
    recorder = RolloutRecorder(simulation, fly.name, fly=fly, timestep=simulation.timestep, simulation_metadata=config.to_mapping())
    recorder.record()
    for _ in range(20):
        action = controller.step()
        apply_locomotion_action(simulation, fly.name, action)
        simulation.step()
        recorder.record()
    dataset = Path('datasets/healthy/Healthy_001')
    exported = export_rollout(recorder.rollout, dataset)
    (dataset / 'logs').mkdir(parents=True, exist_ok=True)
    (dataset / 'logs' / 'run.log').write_text('Healthy_001 bounded rollout completed by Colab notebook 05.\n', encoding='utf-8')
    frames = recorder.rollout.frames
    arrays = {
        'thorax_positions': np.stack([frame.thorax for frame in frames if frame.thorax is not None]),
        'thorax_quaternions': np.stack([frame.orientation for frame in frames if frame.orientation is not None]),
        'time_s': np.asarray([frame.timestamp_s for frame in frames], dtype=float),
        'timestep_s': np.asarray([simulation.timestep], dtype=float),
    }
    com_values = [frame.com for frame in frames]
    if all(value is not None for value in com_values):
        arrays['com_positions'] = np.stack(com_values)
    canonical = dataset / 'rollouts' / 'rollout_arrays.npz'
    canonical.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(canonical, **arrays)
    def digest(path):
        return hashlib.sha256(path.read_bytes()).hexdigest()
    source_commit = subprocess.run(['git', 'rev-parse', 'HEAD'], capture_output=True, text=True, check=False).stdout.strip() or 'unknown'
    declared = [dataset / 'rollouts' / 'rollout_arrays.npz', dataset / 'metadata.json', dataset / 'rollout.json']
    entries = [{'relative_path': path.relative_to(dataset).as_posix(), 'sha256': digest(path), 'byte_size': path.stat().st_size} for path in declared]
    manifest = {
        'schema_version': '1.0', 'dataset_id': 'Healthy_001', 'dataset_type': 'healthy', 'dataset_version': '1.0.0',
        'source_commit': source_commit, 'metadata': 'metadata.json', 'entries': entries,
        'checksums': {entry['relative_path']: entry['sha256'] for entry in entries},
        'citation': 'Drosophila PD FlyGym repository computational rollout package',
        'scientific_scope': 'Computational FlyGym rollout data only; not biological validation.',
    }
    (dataset / 'manifest.json').write_text(json.dumps(manifest, indent=2) + '\n', encoding='utf-8')
    print('Healthy_001 generated:', dataset)
    print('frames:', recorder.rollout.frame_count)
    print('manifest entries:', len(entries))
except Exception as exc:
    print('Healthy_001 generation failed:', type(exc).__name__, exc)